# Splits

Este notebook organiza los frames extraídos en conjuntos de entrenamiento, validación y test usando los splits oficiales del dataset.

**Nota:** DeepFakeDetection se excluye de este split porque utiliza un esquema de nombres de video incompatible con los splits oficiales. Se considera trabajo futuro incorporarla con un split manual.

In [7]:
import json
import shutil
from pathlib import Path
from tqdm.notebook import tqdm
from tqdm.auto import tqdm

FRAMES_PATH = Path('/Users/sofiaramos/Desktop/DEEPFAKES/frames')
SPLITS_PATH = Path('/Users/sofiaramos/Desktop/DEEPFAKES/splits')
OUTPUT_PATH = Path('/Users/sofiaramos/Desktop/DEEPFAKES/dataset_split')

CLASSES = ['real', 'Deepfakes', 'Face2Face', 'FaceShifter', 'FaceSwap', 'NeuralTextures']


## 1. Inspección de los splits oficiales

In [8]:
for split_file in ['train.json', 'val.json', 'test.json']:
    with open(SPLITS_PATH / split_file) as f:
        data = json.load(f)
    print(f'── {split_file} ──')
    print(f'  Pares de videos: {len(data)}')
    print(f'  Ejemplo: {data[:3]}\n')

── train.json ──
  Pares de videos: 360
  Ejemplo: [['071', '054'], ['087', '081'], ['881', '856']]

── val.json ──
  Pares de videos: 70
  Ejemplo: [['720', '672'], ['939', '115'], ['284', '263']]

── test.json ──
  Pares de videos: 70
  Ejemplo: [['953', '974'], ['012', '026'], ['078', '955']]



Cada elemento es un par [video_origen, video_destino]. El video destino es el deepfake generado a partir del video origen. Los splits se definen a nivel de par de videos para garantizar que el mismo sujeto no aparezca en splits distintos, evitando data leakage.

## 2. Verificación de integridad de los splits

In [9]:
splits = {}
for split in ['train', 'val', 'test']:
    with open(SPLITS_PATH / f'{split}.json') as f:
        pairs = json.load(f)
    ids = set()
    for pair in pairs:
        ids.add(pair[0])
        ids.add(pair[1])
    splits[split] = ids

print('Video IDs por split:')
print(f'  Train: {len(splits["train"]):,}')
print(f'  Val:   {len(splits["val"]):,}')
print(f'  Test:  {len(splits["test"]):,}')

print('\nVerificación de solapamiento (debe ser 0 en todos):')
print(f'  Train ∩ Val:  {len(splits["train"] & splits["val"])}')
print(f'  Train ∩ Test: {len(splits["train"] & splits["test"])}')
print(f'  Val ∩ Test:   {len(splits["val"] & splits["test"])}')

Video IDs por split:
  Train: 720
  Val:   140
  Test:  140

Verificación de solapamiento (debe ser 0 en todos):
  Train ∩ Val:  0
  Train ∩ Test: 0
  Val ∩ Test:   0


## 3. Organización de frames por split

In [10]:
def get_video_id(frame_path):
    """Extrae los IDs del video a partir del nombre de la carpeta del frame.
    Ejemplo: frames/Deepfakes/000_003/frame_00010.jpg -> {'000', '003'}
    """
    folder = frame_path.parent.name
    return set(folder.split('_'))

def assign_split(frame_path, splits):
    """Asigna train/val/test a un frame según el ID de su video."""
    video_ids = get_video_id(frame_path)
    for split_name, split_ids in splits.items():
        if video_ids & split_ids:
            return split_name
    return None

total_copied = 0
skipped = 0

for cls in CLASSES:
    cls_path = FRAMES_PATH / cls
    if not cls_path.exists():
        print(f'No se encontró: {cls_path}')
        continue

    all_frames = list(cls_path.rglob('*.jpg'))
    print(f'\n{cls} — {len(all_frames):,} frames')

    for frame_path in tqdm(all_frames, desc=cls):
        split_name = assign_split(frame_path, splits)
        if split_name is None:
            skipped += 1
            continue
        dest = OUTPUT_PATH / split_name / cls / frame_path.parent.name / frame_path.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(frame_path, dest)
        total_copied += 1

print(f'\nFrames copiados: {total_copied:,}')
if skipped:
    print(f'Frames sin split asignado: {skipped:,}')


real — 10,000 frames


real: 100%|██████████| 10000/10000 [00:03<00:00, 3042.58it/s]



Deepfakes — 10,000 frames


Deepfakes: 100%|██████████| 10000/10000 [00:03<00:00, 3034.24it/s]



Face2Face — 10,000 frames


...


FaceSwap — 10,000 frames


FaceSwap: 100%|██████████| 10000/10000 [00:03<00:00, 3224.99it/s]



NeuralTextures — 10,000 frames


NeuralTextures: 100%|██████████| 10000/10000 [00:03<00:00, 3009.69it/s]


Frames copiados: 60,000


## 4. Verificación de la distribución final

In [12]:
grand_total = 0
for split_name in ['train', 'val', 'test']:
    split_path = OUTPUT_PATH / split_name
    if not split_path.exists():
        continue
    print(f'\n  {split_name.upper()}')
    split_total = 0
    for cls in CLASSES:
        cls_path = split_path / cls
        n = sum(1 for _ in cls_path.rglob('*.jpg')) if cls_path.exists() else 0
        print(f'    {cls:<20} {n:>6,} frames')
        split_total += n
    pct = split_total / 60000 * 100
    print(f'    {"TOTAL":<20} {split_total:>6,} frames  ({pct:.1f}%)')
    grand_total += split_total

print(f'  TOTAL GENERAL: {grand_total:,} frames')



  TRAIN
    real                  7,200 frames
    Deepfakes             7,200 frames
    Face2Face             7,200 frames
    FaceShifter           7,200 frames
    FaceSwap              7,200 frames
    NeuralTextures        7,200 frames
    TOTAL                43,200 frames  (72.0%)

  VAL
    real                  1,400 frames
    Deepfakes             1,400 frames
    Face2Face             1,400 frames
    FaceShifter           1,400 frames
    FaceSwap              1,400 frames
    NeuralTextures        1,400 frames
    TOTAL                 8,400 frames  (14.0%)

  TEST
    real                  1,400 frames
    Deepfakes             1,400 frames
    Face2Face             1,400 frames
    FaceShifter           1,400 frames
    FaceSwap              1,400 frames
    NeuralTextures        1,400 frames
    TOTAL                 8,400 frames  (14.0%)
  TOTAL GENERAL: 60,000 frames
